# v1.3 手势分类：Jupyter → TensorFlow → 全 INT8 TFLite

依据 EVSOC 手册 3.4.2、3.4.3。这里是手部裁剪的六类分类器，不能单独定位任意位置的手。保留已训练权重，继续微调5轮。只用训练集真实图片校准，测试集不用于阈值选择。

In [1]:
# v1.3: project-only Jupyter execution; no board programming.
import os,sys,json,hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
ROOT=Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Open this notebook inside the CNN project')
    ROOT=ROOT.parent
assert ROOT.name=='fpga-w.-codex-cnn'
sys.path.insert(0,str(ROOT/'ml/tinyml'))
from workflow import export_int8,predict_tflite,digest
os.environ['KERAS_HOME']=str(ROOT/'ml/tinyml/keras')
tf.keras.utils.set_random_seed(1301)
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))


TensorFlow 2.16.2 GPU []


## 数据与模型
普通卷积、ReLU、Flatten 和全连接均对应厂商示例的已用算子；BN在导出时折叠，Dropout推理关闭。

In [2]:
sys.path.insert(0,str(ROOT/'ml/gesture'))
from train_v1_1 import build,load_split,metrics,LABELS
train_x,train_y=load_split('train');val_x,val_y=load_split('val')
model=build()
previous=tf.keras.models.load_model(ROOT/'ml/gesture/artifacts/best.keras',compile=False)
model.set_weights(previous.get_weights())
model.compile(tf.keras.optimizers.Adam(1e-4),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
out=ROOT/'ml/tinyml/artifacts/classifier';out.mkdir(parents=True,exist_ok=True)
model.summary()
print('Train/validation:',len(train_y),len(val_y))

Model: "gesture_v1_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gray (InputLayer)               │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv0 (Conv2D)                  │ (None, 32, 32, 8)      │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn0 (BatchNormalization)        │ (None, 32, 32, 8)      │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu0 (ReLU)                    │ (None, 32, 32, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 16, 16, 16)     │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 16, 16, 16)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (ReLU)                    │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 8, 8, 24)       │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 8, 8, 24)       │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (ReLU)                    │ (None, 8, 8, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 4, 4, 32)       │         6,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 4, 4, 32)       │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (ReLU)                    │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,990 (58.55 KB)

 Trainable params: 14,830 (57.93 KB)

 Non-trainable params: 160 (640.00 B)

Train/validation: 6150 730


In [3]:
options=tf.data.Options();options.threading.private_threadpool_size=4
aug=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomTranslation(.08,.08,fill_mode='constant',fill_value=128)])
def prepare(x,y):return (aug(tf.cast(x,tf.float32),training=True)-128)/128,y
train_ds=tf.data.Dataset.from_tensor_slices((train_x,train_y)).shuffle(len(train_y),seed=1301).batch(64).map(prepare).with_options(options)
val_ds=tf.data.Dataset.from_tensor_slices(((val_x.astype(np.float32)-128)/128,val_y)).batch(64).with_options(options)
history=model.fit(train_ds,validation_data=val_ds,epochs=5,verbose=2,callbacks=[tf.keras.callbacks.ModelCheckpoint(str(out/'best.keras'),monitor='val_accuracy',mode='max',save_best_only=True),tf.keras.callbacks.CSVLogger(str(out/'training.csv'))])
model=tf.keras.models.load_model(out/'best.keras',compile=False)


Epoch 1/5


C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


97/97 - 3s - 28ms/step - accuracy: 0.8766 - loss: 0.3519 - val_accuracy: 0.9000 - val_loss: 0.2890


Epoch 2/5


97/97 - 1s - 7ms/step - accuracy: 0.8754 - loss: 0.3390 - val_accuracy: 0.9068 - val_loss: 0.2828


...

97/97 - 1s - 7ms/step - accuracy: 0.8816 - loss: 0.3294 - val_accuracy: 0.9068 - val_loss: 0.2799


Epoch 4/5


97/97 - 1s - 7ms/step - accuracy: 0.8886 - loss: 0.3155 - val_accuracy: 0.9068 - val_loss: 0.2799


Epoch 5/5


97/97 - 1s - 7ms/step - accuracy: 0.8865 - loss: 0.3236 - val_accuracy: 0.9137 - val_loss: 0.2709


## 真实样本校准与官方转换
480个训练样本。固定NHWC批次1、INT8输入输出；完整检查算子和中间张量，不只检查文件扩展名。

In [4]:
rng=np.random.default_rng(1301);indices=rng.choice(len(train_x),480,replace=False)
calibration=(train_x[indices].astype(np.float32)-128)/128
path,contract=export_int8(model,calibration,out,'gesture_classifier_int8')
print(json.dumps(contract,indent=2))

C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


{
  "version": "v1.3",
  "sha256": "6da37335b4883c2a7d741df91a6740a2267302fac367324e0d987b789a7ccf56",
  "bytes": 20408,
  "inputs": [
    {
      "name": "gray",
      "shape": [
        1,
        64,
        64,
        1
      ],
      "dtype": "int8",
      "scale": 0.0078125,
      "zero_point": 0
    }
  ],
  "outputs": [
    {
      "name": "Identity",
      "shape": [
        1,
        6
      ],
      "dtype": "int8",
      "scale": 0.12102930247783661,
      "zero_point": 1
    }
  ],
  "ops": [
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "RESHAPE",
    "FULLY_CONNECTED"
  ],
  "calibration_count": 480,
  "calibration_split": "train",
  "full_int8": true,
  "target_micro_invoke_verified": false,
  "hardware_verified": false,
  "vendor_analyzer_returncode": 0
}


In [5]:
vq=predict_tflite(path,(val_x.astype(np.float32)-128)/128)[0]
vp=np.argmax(vq,axis=1);margins=np.sort(vq,axis=1)[:,-1]-np.sort(vq,axis=1)[:,-2]
candidates=[]
for threshold in [0,.5,1,1.5,2,2.5,3,4,5,6]:
    accepted=(vp<5)&(margins>=threshold);correct=accepted&(vp==val_y)
    candidates.append(dict(threshold=threshold,accepted=int(accepted.sum()),precision=float(correct.sum()/max(accepted.sum(),1)),target_recall=float(correct.sum()/max((val_y<5).sum(),1))))
eligible=[r for r in candidates if r['precision']>=.95 and r['accepted']>=30]
selected=max(eligible,key=lambda r:r['target_recall']) if eligible else None
tx,ty=load_split('test');fp=model.predict((tx.astype(np.float32)-128)/128,batch_size=64,verbose=0)
iq=predict_tflite(path,(tx.astype(np.float32)-128)/128)[0]
report=dict(version='v1.3',epochs=5,warm_start=str(ROOT/'ml/gesture/artifacts/best.keras'),model_sha256=digest(out/'best.keras'),tflite_sha256=digest(path),validation=metrics(val_y,vq),threshold_candidates=candidates,selected=selected,float_test=metrics(ty,fp),int8_test=metrics(ty,iq),argmax_agreement=float(np.mean(np.argmax(fp,1)==np.argmax(iq,1))),hardware_verified=False)
(out/'validation_report.json').write_text(json.dumps(report,indent=2))
np.savez(out/'micro_golden.npz',input_float=((tx[:8].astype(np.float32)-128)/128),output_dequantized=iq[:8])
print(json.dumps(report,indent=2))

{
  "version": "v1.3",
  "epochs": 5,
  "warm_start": "C:\\Users\\francis\\Desktop\\fpga-w.-codex-cnn\\ml\\gesture\\artifacts\\best.keras",
  "model_sha256": "a9631519de69d7ece95025271f610836cc95434a1be70656ee124f49fd70f99e",
  "tflite_sha256": "6da37335b4883c2a7d741df91a6740a2267302fac367324e0d987b789a7ccf56",
  "validation": {
    "accuracy": 0.9136986301369863,
    "macro_f1": 0.8962317245869419,
    "recall": {
      "fist": 0.8888888888888888,
      "peace": 0.9444444444444444,
      "palm": 0.9305555555555556,
      "ok": 0.8333333333333334,
      "like": 0.9305555555555556,
      "no_gesture": 0.9216216216216216
    },
    "confusion": [
      [
        64,
        0,
        0,
        3,
        0,
        5
      ],
      [
        0,
        68,
        1,
        2,
        1,
        0
      ],
      [
        0,
        2,
        67,
        2,
        0,
        1
      ],
      [
        1,
        6,
        1,
        60,
        0,
        4
      ],
      [
       

## 部署边界
厂商解析器输出见同目录 vendor_analyzer.log。下一步是 TinyML Generator 参数生成、目标版 TFLite Micro AllocateTensors/Invoke、DDR/摄像头适配和上板验证。PC推理通过不等于这些步骤通过。